# How does your app know two songs share your taste?

Your music app just played you a song you've never heard, by a band you don't know, and it nailed your taste. Somewhere in a data center, two lists of numbers got compared, and the verdict was: same taste.

A song isn't one number. It's loud, fast, sad, bright, a dozen qualities at once, and your taste is that same tangle. Before any code runs, try it yourself: if you had two lists of numbers describing a taste and a song, how would you turn them into one honest match score? Try answering it first. It's genuinely harder than it sounds.

Here's a taste, and two songs: one that actually shares qualities with it, one that just happens to add up to the same total energy.

In [ ]:
import numpy as np

taste = np.array([5.0, 1.0, 0.0, 0.0])
song_aligned = np.array([4.0, 1.0, 0.0, 1.0])
song_mismatched = np.array([0.0, 0.0, 5.0, 1.0])

print('taste:           ', taste)
print('song_aligned:    ', song_aligned)
print('song_mismatched: ', song_mismatched)

Before you run this: try the simplest fix you can think of. Just add up all the numbers in each list and compare the two totals. Predict: will that total tell the aligned song apart from the mismatched one?

In [ ]:
naive_aligned_score = taste.sum() + song_aligned.sum()
naive_mismatched_score = taste.sum() + song_mismatched.sum()

print('naive score, aligned song:   ', round(naive_aligned_score, 2))
print('naive score, mismatched song:', round(naive_mismatched_score, 2))

assert naive_aligned_score == naive_mismatched_score
print('just adding everything up cannot tell these two songs apart')

That naive fix just failed, on purpose. It rewards total energy, not agreement, and it never even looks at which quality goes with which. Here's the move that actually works.

Picture a taste as an arrow instead of a list. Each direction is one quality, some axis like loud or mellow, and the length along that direction is how hard the song leans that way. One probe arrow, that's you, and four candidate arrows, four songs sitting still. Multiply the two arrows entry by entry, then add up the products. That single sum is the score.

In [ ]:
def make_vector(angle_deg, length=1.0):
    angle_rad = np.deg2rad(angle_deg)
    return length * np.array([np.cos(angle_rad), np.sin(angle_rad)])

probe = make_vector(40)
candidates = {
    'c1': make_vector(15),
    'c2': make_vector(35),
    'c3': make_vector(130),
    'c4': make_vector(200),
}

print('probe:', np.round(probe, 2))
for name, vec in candidates.items():
    print(name, np.round(vec, 2))

Before you compute anything: look at the probe and the four candidates above and say which label, c1 through c4, will top out once each one gets dotted against the probe.

In [ ]:
def dot_by_hand(u, v):
    total = 0.0
    for ui, vi in zip(u, v):
        total += ui * vi
    return total

scores = {name: dot_by_hand(probe, vec) for name, vec in candidates.items()}
for name, score in scores.items():
    print(name, 'score:', round(score, 2))

winner = max(scores, key=scores.get)
print('tops out:', winner)

Trust the loop, but check it. Does the hand-rolled version above match NumPy's own dot product for every candidate?

In [ ]:
for name, vec in candidates.items():
    assert np.isclose(dot_by_hand(probe, vec), np.dot(probe, vec))

print('the hand-rolled loop matches np.dot on every candidate')

Before you run this: c1 sits at 15 degrees. As the probe rotates away from that, from 40 degrees up toward 120 degrees, does c1's score fall smoothly the whole way, or does it hit exactly zero somewhere in between? Say where you think zero lands.

In [ ]:
sweep_angles = [40, 60, 80, 105, 120]
c1_vec = candidates['c1']

for angle in sweep_angles:
    swept_probe = make_vector(angle)
    score = np.dot(swept_probe, c1_vec)
    print('probe at', angle, 'deg -> c1 score', round(score, 2))

zero_crossing_score = np.dot(make_vector(105), c1_vec)
assert abs(zero_crossing_score) < 1e-9
print('c1 never moved; only the probe angle did')

Right now c2 tops out, not c1. Before you run this: if you stretch c1's length to 1.3 without rotating it at all, can its raw score pass c2's, even though c2 is still better aligned? Say yes or no.

In [ ]:
c1_stretched = make_vector(15, length=1.3)
new_c1_score = np.dot(probe, c1_stretched)

print('c1 score, length 1.0:', round(scores['c1'], 2))
print('c1 score, length 1.3:', round(new_c1_score, 2))
print('c2 score, unchanged: ', round(scores['c2'], 2))

assert new_c1_score > scores['c2']
print('c1 overtook c2 on length alone; its direction never changed')

## Name the machine you just used

Point the probe along a candidate and the sum climbs. Point it across and the sum dies to zero. Point it against and the sum goes negative. One multiply and one add, and it behaves like a meter with agree, ignore, and oppose on the dial.

**The dot product is a similarity meter.**

You already have the receipts. c1 and c2 came back positive, c3 landed on zero without you asking it to, and c4 came back negative. Same recipe, three readings.

In [ ]:
assert scores['c1'] > 0
assert abs(scores['c3']) < 1e-9
assert scores['c4'] < 0

print('agree  (c1):', round(scores['c1'], 2))
print('ignore (c3):', round(scores['c3'], 2))
print('oppose (c4):', round(scores['c4'], 2))

## Same meter, different aisle

Rebuild it from memory first: two arrows go in, so what two operations turn them into the score? Answer that before reading on.

Now the same meter walks one aisle over. Swap the songs for a job posting: same multiply-and-add, just pointed at career features instead of taste features, the same structure visiting a new aisle, not a new trick. The role wants Python hours, SQL reps, and nights free. In the job aisle, which list plays the probe? You do, the one asking the question.

In [ ]:
you = np.array([8.0, 3.0, 5.0])  # python hours, sql reps, nights free per week
role = np.array([6.0, 4.0, 2.0])  # what the role wants, same three features

fit_score = np.dot(you, role)
print('job fit score:', round(fit_score, 2))

assert np.isclose(fit_score, dot_by_hand(you, role))
print('same multiply-and-add, a different aisle')

## The honest look

A meter you trust blindly is a meter that will lie to you. Point the probe straight along c1's direction and set c1's length back to 1.0 first, so the score reads a clean 1.0.

Before you run this: if you then double c1's length to 2.0, same direction, what will the new score be? Predict the exact number.

In [ ]:
aligned_probe = make_vector(15)
c1_len1 = make_vector(15, length=1.0)
c1_len2 = make_vector(15, length=2.0)

score_len1 = np.dot(aligned_probe, c1_len1)
score_len2 = np.dot(aligned_probe, c1_len2)

print('score at length 1.0:', round(score_len1, 2))
print('score at length 2.0:', round(score_len2, 2))

assert np.isclose(score_len2, 2 * score_len1)
print('doubling the length exactly doubled the score')

Doubling the length exactly doubled the score. The meter is linear in each arrow you feed it, and that clean scaling is the crack: a long, loud song can win on sheer size, not on agreement. A shout beats a whisper even when the whisper agrees with you more.

## For the walk home

Next time your app hands you a stranger's song that's exactly right, you'll know the number underneath the verdict, the same multiply-and-add you just rebuilt by hand. You'll also know that number can be shouted over by size alone.

What is the cheapest fix that keeps the direction and forgets the size?

One thing to try next: write a function that rescales a vector before scoring it, then test it against the length-cheat cell above. Does your fix stop c1 from overtaking c2 the same way it did there?